# Chương 2: Hệ thống Khuyến nghị Phim — FAISS + LLM Re-rank

## Pipeline tổng quan

```
User query → NLU (query_vector 256d + entities + hard_filters)
                ↓
    Query Embedding Construction
    (plot 256d + genre 19d + cast 256d → weighted concat 531d)
                ↓
    FAISS IndexFlatIP → Top-100 candidates
                ↓
    Hard Filter (year, genre, person) → Top-20
                ↓
    Ollama LLM Re-rank → Top-5 final results
                ↓
    HTML cards response
```

## Các bước thực hiện:
1. **Cài đặt dependencies** (`faiss-cpu`, `tqdm`)
2. **Cài Ollama + pull model** (`gemma3:4b`)
3. **Build offline database** — Crawl 5000+ phim từ TMDB → PhoBERT embeddings → FAISS index
4. **Chạy app** — Auto-detect offline DB, dùng FAISS pipeline hoặc fallback TMDB API

## Bước 1: Cài đặt Dependencies

In [1]:
!pip install faiss-cpu tqdm -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Kiểm tra import
import faiss
import tqdm
print(f"faiss version: {faiss.__version__ if hasattr(faiss, '__version__') else 'OK'}")
print(f"tqdm version: {tqdm.__version__}")

faiss version: 1.13.2
tqdm version: 4.67.3


## Bước 2: Cài đặt Ollama + Pull model

> **Yêu cầu**: Ollama phải được cài đặt trên máy. Tải tại: https://ollama.com/download
>
> Sau khi cài, chạy cell dưới để pull model `gemma3:4b` (~3GB).
> Nếu Ollama chưa cài, hệ thống vẫn hoạt động bình thường — chỉ bỏ qua bước LLM re-rank.

In [3]:
import subprocess, requests

# Kiểm tra Ollama đã cài chưa
try:
    resp = requests.get("http://localhost:11434/api/tags", timeout=3)
    if resp.status_code == 200:
        models = [m["name"] for m in resp.json().get("models", [])]
        print(f"✅ Ollama đang chạy. Models hiện có: {models}")
        if not any("gemma3" in m for m in models):
            print("⏳ Đang pull model gemma3:4b... (có thể mất vài phút)")
            result = subprocess.run(
                ["ollama", "pull", "gemma3:4b"],
                capture_output=True, text=True, timeout=600
            )
            print(result.stdout if result.returncode == 0 else f"❌ Lỗi: {result.stderr}")
        else:
            print("✅ Model gemma3 đã có sẵn!")
    else:
        print("⚠️ Ollama server phản hồi bất thường")
except requests.ConnectionError:
    print("⚠️ Ollama chưa chạy hoặc chưa cài đặt.")
    print("   Tải tại: https://ollama.com/download")
    print("   Sau khi cài, chạy 'ollama serve' rồi chạy lại cell này.")
    print("   → Hệ thống vẫn hoạt động bình thường, chỉ bỏ qua LLM re-rank.")
except Exception as e:
    print(f"⚠️ Lỗi: {e}. Bỏ qua Ollama — hệ thống vẫn hoạt động.")

⚠️ Ollama chưa chạy hoặc chưa cài đặt.
   Tải tại: https://ollama.com/download
   Sau khi cài, chạy 'ollama serve' rồi chạy lại cell này.
   → Hệ thống vẫn hoạt động bình thường, chỉ bỏ qua LLM re-rank.


## Bước 3: Build Offline Movie Database

Pipeline: **Crawl TMDB** (5000+ phim) → **PhoBERT embeddings** (plot + genre + cast = 531d) → **FAISS index**

> ⚠️ **Quan trọng**: Cần set `TMDB_API_KEY` hoặc `TMDB_READ_TOKEN` trước khi chạy.
> Thời gian crawl phụ thuộc vào target và tốc độ mạng.

In [4]:
import sys, os
from pathlib import Path

# Setup paths
BASE_DIR = Path(r"d:\desktop\test\movie_chatbot_nlu")
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# Load env vars từ .env nếu có
env_path = BASE_DIR.parent / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

print(f"TMDB_API_KEY set: {'✅' if os.environ.get('TMDB_API_KEY') else '❌'}")
print(f"TMDB_READ_TOKEN set: {'✅' if os.environ.get('TMDB_READ_TOKEN') else '❌'}")

TMDB_API_KEY set: ✅
TMDB_READ_TOKEN set: ❌


In [5]:
from src.api_client import TMDBClient
from src.movie_embedder import MovieEmbedder
from src.movie_database import MovieDatabase
from src.config import MOVIE_DB_DIR, RECOMMENDATION_CONFIG

print(f"Movie DB directory: {MOVIE_DB_DIR}")
print(f"Embedding dim: {RECOMMENDATION_CONFIG['embedding_dim']}")
print(f"Crawl target: {RECOMMENDATION_CONFIG['tmdb_crawl_target']}")
print(f"FAISS top-K: {RECOMMENDATION_CONFIG['faiss_top_k']}")
print(f"LLM rerank top-K: {RECOMMENDATION_CONFIG['llm_rerank_top_k']}")
print(f"Embedding weights: {RECOMMENDATION_CONFIG['embedding_weights']}")

Config loaded OK (Semantic Parsing + SimCSE)
PHOBERT_MODEL   : vinai/phobert-base-v2
INTENT_LABELS   : ['find_movie', 'recommendation', 'movie_info', 'person_info', 'genre_filter', 'greeting', 'goodbye', 'out_of_scope']
ALL_SLOT_NAMES  : ['genre', 'like_movie', 'name', 'person', 'title', 'year']
NUM_SLOTS       : 6
FRAME_SCHEMA    : 8 frames
MOVIE_DB_DIR    : d:\desktop\test\movie_chatbot_nlu\data\movies
EMBEDDING_DIM   : 531


c:\Users\LENOVO\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Movie DB directory: d:\desktop\test\movie_chatbot_nlu\data\movies
Embedding dim: 531
Crawl target: 5000
FAISS top-K: 100
LLM rerank top-K: 5
Embedding weights: {'plot': 0.5, 'genre': 0.3, 'cast': 0.2}


### 3.1 Crawl phim từ TMDB + Encode embeddings + Build FAISS index

> Đổi `TARGET` nếu muốn crawl ít hơn để test nhanh (vd: 100).
> Full 5000 phim mất khoảng 20-30 phút do rate limit TMDB API.

In [ ]:
TARGET = 500  # Dùng 500 để demo nhanh (~5 phút). Đổi thành 5000 cho production.
print(f"Target: {TARGET} movies")

# Khởi tạo components
client = TMDBClient()
embedder = MovieEmbedder()
db = MovieDatabase()

# Build: crawl → embed → FAISS index → save to disk
db.build(embedder, client, target=int(TARGET))

Target: 500 movies
[TMDBClient] Dang dung api_key query param


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6734.24it/s]
RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[MovieEmbedder] Ready on cuda | plot=256d + genre=19d + cast=256d = 531d
[MovieDatabase] Building with target=500 movies...
Starting crawl: target=500 movies, 30 pages
Discovered 600 unique movies from 30 pages
Step 1 done: 500 movies discovered
Step 2: Getting details for 500 movies...
  Details: 100/500
  Details: 200/500
  Details: 300/500
  Details: 400/500
  Details: 500/500
Crawl complete: 500 movies with full details
  Crawled 500 movies with details
  Extracted metadata for 500 movies


Encoding movies:   7%|▋         | 33/500 [00:00<00:11, 41.64it/s]

### 3.2 Kiểm tra kết quả build

In [ ]:
import os

print(f"=== Offline Movie Database ===")
print(f"Số phim: {len(db.movies)}")
print(f"Embeddings shape: {db.embeddings.shape}")
print(f"FAISS index size: {db.index.ntotal} vectors")
print(f"DB loaded: {db.is_loaded}")
print()

# Hiển thị files trên disk
for f in os.listdir(MOVIE_DB_DIR):
    fpath = os.path.join(MOVIE_DB_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {f}: {size_mb:.1f} MB")

print()
# Sample 5 phim đầu
print("=== 5 phim đầu tiên ===")
for m in db.movies[:5]:
    genres = ", ".join(m.get("genre_names", []))
    cast = ", ".join(m.get("cast_names", [])[:3])
    print(f"  [{m['id']}] {m['title']} ({m['year']}) - {genres} | Cast: {cast}")

## Bước 4: Test FAISS Search + Hard Filter + LLM Re-rank

Thử nghiệm pipeline trực tiếp trước khi tích hợp vào app.

In [ ]:
import numpy as np

# --- Test 1: FAISS search bằng text query ---
print("=== Test 1: FAISS search bằng text ===")
query_text = "phim hành động có nhiều cảnh đánh nhau"
qv = embedder.encode_query(
    query_vector=embedder.encode_plot(query_text),
    genre_ids=[28],  # Action
)
results = db.search(qv, k=10)
print(f"Query: '{query_text}'")
print(f"Top 10 results:")
for i, (m, score) in enumerate(results):
    genres = ", ".join(m.get("genre_names", []))
    print(f"  {i+1}. [{score:.4f}] {m['title']} ({m['year']}) - {genres}")

print()

# --- Test 2: Search + Hard filter ---
print("=== Test 2: FAISS + Hard filter (genre=Horror, year=2023) ===")
qv2 = embedder.encode_query(
    query_vector=embedder.encode_plot("phim kinh dị rùng rợn"),
    genre_ids=[27],  # Horror
)
candidates = db.search(qv2, k=50)
filtered = db.hard_filter(candidates, {"GENRE": "kinh dị", "YEAR": "2023"})
print(f"Before filter: {len(candidates)} | After filter: {len(filtered)}")
for i, (m, score) in enumerate(filtered[:5]):
    genres = ", ".join(m.get("genre_names", []))
    print(f"  {i+1}. [{score:.4f}] {m['title']} ({m['year']}) - {genres}")

In [ ]:
# --- Test 3: LLM Re-rank (nếu Ollama available) ---
print("=== Test 3: LLM Re-rank ===")
from src.llm_reranker import OllamaReranker

reranker = OllamaReranker()
if reranker.is_available():
    print(f"✅ Ollama available, model: {reranker.model}")
    # Lấy top-20 candidates cho re-rank
    qv3 = embedder.encode_query(
        query_vector=embedder.encode_plot("phim lãng mạn cho cặp đôi xem cuối tuần"),
        genre_ids=[10749],  # Romance
    )
    candidates = db.search(qv3, k=50)
    candidates = candidates[:20]
    
    reranked = reranker.rerank(
        "phim lãng mạn cho cặp đôi xem cuối tuần",
        candidates,
        top_k=5
    )
    print(f"\nTop 5 sau re-rank:")
    for i, (m, score) in enumerate(reranked):
        reason = m.get("rerank_reason", "")
        print(f"  {i+1}. {m['title']} ({m['year']})")
        if reason:
            print(f"     → {reason}")
else:
    print("⚠️ Ollama không available. Bỏ qua LLM re-rank.")
    print("   Hệ thống vẫn hoạt động tốt với FAISS + hard filter.")

## Bước 5: Test tích hợp đầy đủ — Recommendation Engine

Khởi tạo `TMDBRecommendationEngine` — tự động detect offline DB + Ollama.
Nếu DB đã build ở bước 3 thì dùng FAISS pipeline, không thì fallback TMDB API.

In [ ]:
from src.recommendation import TMDBRecommendationEngine

engine = TMDBRecommendationEngine()
print(f"FAISS pipeline: {'✅ Active' if engine._use_faiss else '❌ Fallback to API'}")
print(f"LLM reranker: {'✅ Available' if engine.reranker and engine.reranker.is_available() else '⚠️ Disabled'}")

In [ ]:
from IPython.display import HTML, display

# Giả lập NLU output để test engine.handle()
test_queries = [
    {
        "name": "Tìm phim hành động",
        "nlu_result": {
            "input": "tìm phim hành động hay",
            "intent": "find_movie",
            "entities": {"GENRE": ["Phim Hành Động"]},
            "hard_filters": {"GENRE": "hành động"},
            "query_vector": embedder.encode_plot("tìm phim hành động hay").tolist(),
        },
    },
    {
        "name": "Gợi ý phim kinh dị 2023",
        "nlu_result": {
            "input": "gợi ý phim kinh dị năm 2023",
            "intent": "recommendation",
            "entities": {"GENRE": ["Phim Kinh Dị"], "YEAR": ["2023"]},
            "hard_filters": {"GENRE": "kinh dị", "YEAR": "2023"},
            "query_vector": embedder.encode_plot("gợi ý phim kinh dị năm 2023").tolist(),
        },
    },
    {
        "name": "Phim lãng mạn",
        "nlu_result": {
            "input": "phim tình cảm lãng mạn xem cuối tuần",
            "intent": "genre_filter",
            "entities": {"GENRE": ["Phim Lãng Mạn"]},
            "hard_filters": {"GENRE": "lãng mạn"},
            "query_vector": embedder.encode_plot("phim tình cảm lãng mạn xem cuối tuần").tolist(),
        },
    },
]

for test in test_queries:
    print(f"\n{'='*60}")
    print(f"📝 Query: {test['name']}")
    print(f"   Intent: {test['nlu_result']['intent']}")
    print(f"{'='*60}")
    
    result = engine.handle(test["nlu_result"], context={})
    print(f"\n💬 {result['text']}")
    if result.get("cards"):
        display(HTML(result["cards"]))

## Bước 6: Chạy App (Gradio UI)

Chạy cell dưới để khởi động Gradio web UI. App tự động dùng FAISS pipeline nếu offline DB đã build.

> Mở browser tại link hiển thị (thường là `http://127.0.0.1:7860`)

In [ ]:
%run d:/desktop/test/app.py